# FTIR evaluation — portable JupyterLab notebook

Batch analysis of FTIR spectra using a five-Gaussian model for the Amide I region.

**Folder layout:** Place this notebook in the repository root, put tab-separated `.txt` spectra in `data/`, and run all cells. Results are saved to `Evaluated/` (created automatically). No computer-specific paths are required.

**Scientific caution:** Fixed band centers follow the original D₂O-oriented model. Relative fitted areas are not validated secondary-structure percentages. Inspect fit residuals and spectra before interpretation.

**Attribution:** Adapted from https://github.com/anastasia-ragulskaya/FTIR_code . Verify the original licensing/redistribution permissions before publishing adapted code.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from IPython.display import display

%matplotlib inline

# Relative to the directory containing this notebook when JupyterLab starts here.
project_folder = Path.cwd()
input_folder = project_folder / "data"
output_folder = project_folder / "Evaluated"
output_folder.mkdir(parents=True, exist_ok=True)

if not input_folder.is_dir():
    raise FileNotFoundError(
        f"Create the input folder '{input_folder}' and put your TXT files inside it."
    )

files = sorted(input_folder.glob("*.txt"))
print(f"Found {len(files)} TXT files in {input_folder}")
if not files:
    print("No TXT files found. Add spectra to the data folder and rerun.")


In [ ]:
# Original workflow functions, made self-contained for this notebook.
def import_data(text):
    x, y = [], []
    for line_number, line in enumerate(text, 1):
        if not line.strip():
            continue
        columns = line.strip().split("\t")
        if len(columns) < 2:
            raise ValueError(f"Line {line_number}: expected two tab-separated columns")
        x.append(float(columns[0]))
        y.append(float(columns[1]))
    return x, y

def subtract_background(datax_amid1, datay_amid1):
    x = np.asarray(datax_amid1, dtype=float)
    y = np.asarray(datay_amid1, dtype=float)
    if len(x) < 2 or x[0] == x[-1]:
        raise ValueError("Need two distinct baseline endpoints")
    baseline = y[0] + (y[-1] - y[0]) * (x - x[0]) / (x[-1] - x[0])
    return y - baseline

centers = np.array([1652, 1630, 1679, 1671, 1645])
labels = ["Alpha helix", "Beta sheet 1630", "Beta sheet 1679",
          "Loop", "Disordered"]

def gaussian_components(x, parameters):
    x = np.asarray(x, dtype=float)
    a = np.asarray(parameters[:5])
    s = np.asarray(parameters[5:])
    return np.array([
        a[i] * np.exp(-((x - centers[i]) ** 2) / (2 * s[i] ** 2))
        for i in range(5)
    ])

def fit_gauss5(x, y, bounds=(
    (0, 0, 0, 0, 0, 0, 0, 0, 0, 0),
    (1, 1, 1, 1, 1, 100, 100, 100, 100, 100)
)):
    def model(x, *params):
        return gaussian_components(x, params).sum(axis=0)
    parameters, covariance = curve_fit(
        model, np.asarray(x, dtype=float), np.asarray(y, dtype=float),
        bounds=bounds, p0=[0.1] * 5 + [10.0] * 5, maxfev=20000
    )
    errors = np.sqrt(np.diag(covariance))
    return model(x, *parameters), parameters, errors


In [ ]:
def save_graphs(sample, graph_folder, x, y, xa, ya, baseline,
                corrected, params, fitted, residuals, r_squared, rmse):
    graph_folder.mkdir(parents=True, exist_ok=True)

    def save(fig, name):
        fig.tight_layout()
        fig.savefig(graph_folder / name, dpi=300, bbox_inches="tight")
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, linewidth=1)
    ax.invert_xaxis()
    ax.set(title=f"{sample} — Original FTIR", xlabel="Wavenumber (cm⁻¹)",
           ylabel="Absorbance")
    save(fig, "01_Original_FTIR.png")

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(xa, ya)
    ax.set(xlim=(1700, 1600), title=f"{sample} — Amide I",
           xlabel="Wavenumber (cm⁻¹)", ylabel="Absorbance")
    save(fig, "02_Amide_I.png")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].plot(xa, ya, label="Original")
    axes[0].plot(xa, baseline, "--", label="Baseline")
    axes[0].legend()
    axes[0].set_title("Before correction")
    axes[1].plot(xa, corrected)
    axes[1].axhline(0, color="gray", linestyle="--")
    axes[1].set_title("After correction")
    for ax in axes:
        ax.set(xlim=(1700, 1600), xlabel="Wavenumber (cm⁻¹)",
               ylabel="Absorbance")
    save(fig, "03_Baseline_Correction.png")

    xs = np.linspace(1600, 1700, 1000)
    components = gaussian_components(xs, params)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.plot(xa, corrected, "ko", markersize=2, label="Experimental")
    for label, component in zip(labels, components):
        ax.plot(xs, component, "--", label=label)
    ax.plot(xs, components.sum(axis=0), "r-", linewidth=2, label="Total fit")
    ax.set(xlim=(1700, 1600), title=f"{sample} — Gaussian fit",
           xlabel="Wavenumber (cm⁻¹)", ylabel="Corrected absorbance")
    ax.legend(fontsize=8)
    save(fig, "04_Gaussian_Deconvolution.png")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(xa, residuals)
    ax.axhline(0, color="gray", linestyle="--")
    ax.set(xlim=(1700, 1600),
           title=f"{sample} — Residuals (R²={r_squared:.3f}, RMSE={rmse:.5g})",
           xlabel="Wavenumber (cm⁻¹)", ylabel="Residual")
    save(fig, "05_Fit_Residuals.png")


In [ ]:
summary, failed = [], []

for number, filename in enumerate(files, start=1):
    print(f"[{number}/{len(files)}] {filename.name}")
    try:
        with filename.open("r", encoding="utf-8-sig") as handle:
            datax, datay = import_data(handle)
        x = np.asarray(datax, dtype=float)
        y = np.asarray(datay, dtype=float)
        valid = np.isfinite(x) & np.isfinite(y)
        x, y = x[valid], y[valid]

        mask = (x >= 1600) & (x <= 1700)
        xa, ya = x[mask], y[mask]
        order = np.argsort(xa)
        xa, ya = xa[order], ya[order]
        if len(np.unique(xa)) < 15:
            raise ValueError("Fewer than 15 unique Amide I wavenumbers")
        if len(np.unique(xa)) != len(xa):
            raise ValueError("Duplicate Amide I wavenumbers detected")

        corrected = np.asarray(subtract_background(xa, ya))
        baseline = ya - corrected
        fitted, parameters, errors = fit_gauss5(xa, corrected)
        if not np.all(np.isfinite(parameters)):
            raise ValueError("Non-finite fit parameters")

        areas = parameters[:5] * parameters[5:] * np.sqrt(2 * np.pi)
        if areas.sum() <= 0:
            raise ValueError("Invalid total fitted area")
        percentages = 100 * areas / areas.sum()
        residuals = corrected - fitted
        rmse = np.sqrt(np.mean(residuals ** 2))
        ss_tot = np.sum((corrected - corrected.mean()) ** 2)
        r_squared = 1 - np.sum(residuals ** 2) / ss_tot if ss_tot > 0 else np.nan

        fit_df = pd.DataFrame({
            "Secondary structure": labels,
            "Peak position (cm-1)": centers,
            "Amplitude": parameters[:5], "Amplitude SE": errors[:5],
            "Sigma (cm-1)": parameters[5:], "Sigma SE": errors[5:]
        })
        area_df = pd.DataFrame({
            "Secondary structure": labels,
            "Peak position (cm-1)": centers,
            "Gaussian area": areas, "Relative area (%)": percentages
        })
        spectrum_df = pd.DataFrame({
            "Wavenumber": xa, "Original absorbance": ya,
            "Baseline": baseline, "Corrected absorbance": corrected,
            "Gaussian fit": fitted, "Residual": residuals
        })
        excel_file = output_folder / f"{filename.stem}_Gaussian_fit.xlsx"
        with pd.ExcelWriter(excel_file, engine="openpyxl") as writer:
            fit_df.to_excel(writer, sheet_name="Fitting Parameters", index=False)
            area_df.to_excel(writer, sheet_name="Peak Areas", index=False)
            spectrum_df.to_excel(writer, sheet_name="Spectrum", index=False)

        save_graphs(filename.stem, output_folder / filename.stem,
                    x, y, xa, ya, baseline, corrected,
                    parameters, fitted, residuals, r_squared, rmse)

        summary.append({
            "Sample": filename.stem,
            "Alpha helix (%)": percentages[0],
            "Beta sheet 1630 (%)": percentages[1],
            "Beta sheet 1679 (%)": percentages[2],
            "Total beta sheet (%)": percentages[1] + percentages[2],
            "Loop (%)": percentages[3],
            "Disordered (%)": percentages[4],
            "R squared": r_squared, "RMSE": rmse,
            "Unique wavenumbers": len(xa)
        })
        print("  Completed")
    except Exception as exc:
        failed.append({"Sample": filename.name, "Error": str(exc)})
        print(f"  FAILED: {exc}")

summary_df = pd.DataFrame(summary)
failed_df = pd.DataFrame(failed)
with pd.ExcelWriter(output_folder / "FTIR_All_Samples_Summary.xlsx",
                    engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    failed_df.to_excel(writer, sheet_name="Failed Samples", index=False)

print(f"Finished: {len(summary)} successful; {len(failed)} failed")
display(summary_df.round(3))
